# AutoGluon AutoML — predykcja anulowania rezerwacji

Notebook porównuje automatycznie dobrane modele AutoGluon z ręcznie
przygotowanym pipeline'em Kedro. Używa tego samego `model_input`, podziału
80/20, `random_state=42` i nietkniętego zbioru testowego.

## Kontrola zasobów

- `time_limit=300` ogranicza trening AutoGluon do około 5 minut.
- `presets="medium_quality"` pomija kosztowne wielowarstwowe bagging/stacking.
- modele i raporty trafiają do ignorowanych katalogów `data/06_models/` oraz
  `data/08_reporting/`.

Lokalnie notebook należy uruchamiać w osobnym środowisku:

```powershell
python -m venv .venv-autogluon
.venv-autogluon\Scripts\python.exe -m pip install -r requirements-autogluon.txt
.venv-autogluon\Scripts\jupyter-nbconvert.exe --execute --inplace notebooks/02_autogluon.ipynb
```


In [1]:
from pathlib import Path
import json

import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

TARGET = "is_canceled"
TEST_SIZE = 0.2
RANDOM_STATE = 42
TIME_LIMIT = 300
MODEL_INPUT = ROOT / "data" / "05_model_input" / "model_input.parquet"
MODEL_PATH = ROOT / "data" / "06_models" / "autogluon"
REPORT_PATH = ROOT / "data" / "08_reporting" / "autogluon_metrics.json"

MODEL_INPUT


C:\Users\damia\Desktop\MAIN\PJATK\eightSemester\ASI\ASI_projekt_koncowy\.venv-autogluon\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WindowsPath('C:/Users/damia/Desktop/MAIN/PJATK/eightSemester/ASI/ASI_projekt_koncowy/data/05_model_input/model_input.parquet')

## 1. Dane i uczciwy podział

Notebook korzysta z macierzy cech wygenerowanej przez pipeline Kedro. Dzięki
temu AutoGluon i modele ręczne otrzymują dokładnie te same dane wejściowe.
Przed uruchomieniem notebooka wykonaj `kedro run --pipeline data_processing`.


In [2]:
data = pd.read_parquet(MODEL_INPUT)
train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=data[TARGET],
)

print(f"Train: {train_data.shape}, test: {test_data.shape}")
print(train_data[TARGET].value_counts(normalize=True).sort_index())


Train: (95368, 83), test: (23842, 83)
is_canceled
0    0.629236
1    0.370764
Name: proportion, dtype: float64


## 2. AutoML

AutoGluon sam dobiera algorytmy i ich konfiguracje. Metryką wyboru jest
`roc_auc`, zgodnie z porównaniem modeli i strojeniem Optuną.


In [3]:
predictor = TabularPredictor(
    label=TARGET,
    eval_metric="roc_auc",
    path=str(MODEL_PATH),
).fit(
    train_data=train_data,
    time_limit=TIME_LIMIT,
    presets="medium_quality",
)


Verbosity: 2 (Standard Logging)


=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.7
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Memory Avail:       1.88 GB / 15.80 GB (11.9%)
Disk Space Avail:   38.99 GB / 456.17 GB (8.5%)


Presets specified: ['medium_quality']


Using hyperparameters preset: hyperparameters='default'


Beginning AutoGluon training ... Time limit = 300s


...

	0.9499	 = Validation score   (roc_auc)


	0.09s	 = Training   runtime


	0.0s	 = Validation runtime


AutoGluon training complete, total runtime = 56.22s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 5833.4 rows/s (2500 batch size)


TabularPredictor saved. To load, use: predictor = TabularPredictor.load("C:\Users\damia\Desktop\MAIN\PJATK\eightSemester\ASI\ASI_projekt_koncowy\data\06_models\autogluon")


## 3. Leaderboard i ewaluacja na nietkniętym zbiorze testowym

In [4]:
leaderboard = predictor.leaderboard(test_data, silent=True)
leaderboard


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.956156,0.949900,roc_auc,3.310216,0.428570,41.570884,0.069596,0.000000,0.088250,2,True,5
1,RandomForestEntr,0.956086,0.949133,roc_auc,0.874352,0.213936,11.993002,0.874352,0.213936,11.993002,1,True,2
2,RandomForestGini,0.955063,0.947885,roc_auc,0.674121,0.069748,10.016746,0.674121,0.069748,10.016746,1,True,1
3,ExtraTreesEntr,0.951978,0.946575,roc_auc,0.874874,0.071888,10.362296,0.874874,0.071888,10.362296,1,True,4
4,ExtraTreesGini,0.951317,0.946786,roc_auc,0.817273,0.072998,9.110589,0.817273,0.072998,9.110589,1,True,3


In [5]:
X_test = test_data.drop(columns=[TARGET])
y_test = test_data[TARGET]
y_pred = predictor.predict(X_test)
y_proba = predictor.predict_proba(X_test, as_multiclass=False)

metrics = {
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "f1": round(f1_score(y_test, y_pred), 4),
    "roc_auc": round(roc_auc_score(y_test, y_proba), 4),
    "best_model": predictor.model_best,
    "time_limit_seconds": TIME_LIMIT,
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
metrics


{'accuracy': 0.8889,
 'f1': 0.8438,
 'roc_auc': 0.9562,
 'best_model': 'WeightedEnsemble_L2',
 'time_limit_seconds': 300}

## Wnioski

Leaderboard pokazuje, które rodziny modeli AutoGluon przetestował w zadanym
budżecie. Wynik należy zestawić z ręcznie przygotowanym Random Forest po
Optunie. AutoML jest mocnym benchmarkiem, ale ręczny pipeline pozostaje
łatwiejszy do kontroli, wyjaśnienia i wdrożenia.
